# Countdown SFT on program-written search traces

Fine-tuning to seed reasoning behaviour and format adherence T. Prompt = the GRPO prompt
(puzzle + format block). Qwen3: thinking off (enable_thinking=False) in training (TRL's per-example
chat_template_kwargs) and in every generation here.

After training, [grade] samples S's grade puzzles (200 four-number + 200 three-number, 8 x T 0.9, cap 2,048, read at
1,024 and 2,048) with the adapter, in the layout of Consistency_Countdown_Screen_Qwen (whose untrained rows it prints
alongside).


In [ ]:
# [config]
import os
MODELS = {   
    "qwen3-0.6b": ("Qwen/Qwen3-0.6B", {"enable_thinking": False}),
    "qwen3-1.7b": ("Qwen/Qwen3-1.7B", {"enable_thinking": False}),
}
MODEL_KEY = "qwen3-0.6b"
MODEL, CHAT_KWARGS = MODELS[MODEL_KEY]
SEED = 0
# SFT: as the Llama start (Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500)
N_SFT, N_SFT_3 = 500, 500        # 4- and 3-number traces
N_SFT_VAL = 100                  # per size: validation loss; greedy accuracy on the first 50 of each size
N_JUDGE_SFT = 0                  # gold judge-format examples mixed in (the Llama start: 500); 0 = traces only
SFT_EPOCHS, SFT_LR = 1, 1e-4
LORA_R, LORA_ALPHA = 16, 32
SFT_BATCH, SFT_GRAD_ACC = 4, 4   # effective batch 16: (500 + 500) / 16 = 63 optimizer steps
SFT_EVAL_STEPS = 25
SFT_ACC_EVAL_STEPS = 25          # greedy accuracy on the validation puzzles (and at step 0 = the untrained model)
RETRAIN = False
TRACE_VERSION = 4
FAIL_SHARES = (0.76, 0.16, 0.08)            # traces with 0 / 1 / 2 caught false finishes
FAILS_BEFORE_WIN = (0.35, 0.3, 0.25, 0.1)   # traces winning on attempt 1 / 2 / 3 / 4
N_IMPOSSIBLE, N_IMPOSSIBLE_3 = 0, 0
# Grade: S's grade puzzles, SAMPLES x T, one generation at MAX_TOKENS read at CAPS
N_GRADE, GRADE_SAMPLES, TEMPERATURE = 200, 8, 0.9
MAX_TOKENS, CAPS = 2048, (1024, 2048)
VLLM_GPU_MEMORY_UTILIZATION = 0.42
UNASSIGN_WHEN_DONE = False       # [release]: free the Colab runtime after [grade]

DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
N_PUZZLES = 200                  # the test draw's 4-number puzzles the SFT puzzles avoid (= S's grade puzzles)
SFT_RUN_NAME = (f"{MODEL.split('/')[-1]}-countdown{N_SFT}-ep{SFT_EPOCHS}-lr{SFT_LR:g}-seed{SEED}-tr{TRACE_VERSION}"
                + f"-f{round(100 * FAIL_SHARES[1])}-{round(100 * FAIL_SHARES[2])}"
                + f"-w{'-'.join(str(round(100 * s)) for s in FAILS_BEFORE_WIN)}"
                + f"-imp{N_IMPOSSIBLE}x{N_IMPOSSIBLE_3}"
                + (f"-judge{N_JUDGE_SFT}" if N_JUDGE_SFT else "") + (f"-mix3x{N_SFT_3}" if N_SFT_3 else ""))
SFT_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN_NAME}"
SFT_ADAPTER_DIR = f"{SFT_DIR}/adapter"
S_GRADE = f"{DRIVE}/countdown/grpo/olmo3-7b-format50-cap1024-seed0/grade"   # S's grade files: the grade puzzles
GRADE_FILES = {n: f"{S_GRADE}/gen_Olmo-3-7B-Instruct-format50_user_n{n}.csv" for n in (4, 3)}
GRADE_DIR = f"{SFT_DIR}/grade_n{N_GRADE}x{GRADE_SAMPLES}_t{TEMPERATURE:g}_cap{MAX_TOKENS}_seed{SEED}"
SCREEN = f"{DRIVE}/countdown/screen/qwen3_grade_n200x8_t0.9_cap2048_seed0/summary.csv"   # the untrained rows
assert TRACE_VERSION == 4 and not (N_IMPOSSIBLE or N_IMPOSSIBLE_3)
print(f"MODEL={MODEL} {CHAT_KWARGS or ''}\nSFT_DIR={SFT_DIR}\nGRADE_DIR={GRADE_DIR}")


MODEL=Qwen/Qwen3-0.6B {'enable_thinking': False}
SFT_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500
GRADE_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/grade_n200x8_t0.9_cap2048_seed0


In [2]:
# [drive + HF] Mount Drive, HF login (Colab secret HF_TOKEN), and the files the notebook needs.
import os
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
for f in ("countdown.py", "prompts.py", "rewards.py", "traces.py", "judge_data.py"):
    assert os.path.exists(f"{CODE_DIR}/{f}"), f"missing {CODE_DIR}/{f}"
for f in GRADE_FILES.values():
    assert os.path.exists(f), f"missing {f}"
print("adapter:", "exists, [SFT train] skips" if os.path.exists(f"{SFT_ADAPTER_DIR}/adapter_config.json") else "none yet")


adapter: none yet


In [3]:
# [install] As Consistency_GRPO_Format. Once per runtime, then restart the session (Runtime -> Restart session) and
# continue with [config], [drive + HF], [vLLM setup].
%pip install -U "trl[vllm]" peft
%pip uninstall -y torchao


In [ ]:
# [vLLM setup]
import os, sys
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the vLLM engine in this process
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):   # Colab's stdout/stderr have no OS file descriptor
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)
import importlib.metadata as md
for p in ["torch", "transformers", "trl", "peft", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")


name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB

torch          2.13.0
transformers   5.19.0
trl            1.17.1
peft           0.21.2
datasets       4.8.5
vllm           0.32.0


In [5]:
# [setup] Tokenizer (CHAT_KWARGS on every template call), the Countdown code (Drive), the checks [SFT train] uses (tag,
# diagnose) and the screen's labels (label) for [grade].
import sys, re, json, random, importlib, time, gc
from collections import Counter
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None or tok.pad_token_id == tok.eos_token_id:
    raise ValueError(f"{MODEL}: needs a pad token other than EOS ({tok.pad_token!r})")

def chat_text(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_KWARGS)

def chat_ids(messages):
    return tok(chat_text(messages), add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)
import prompts, countdown, traces, judge_data, rewards as R
for _m in (prompts, countdown, traces, judge_data, R):   # prompts first: the others import from it
    importlib.reload(_m)
from countdown import check_answer
from prompts import countdown_prompt, countdown_question
from traces import make_trace
tag = R.tag
diagnose = lambda answer, numbers, target, truncated: R.diagnose(answer or None, numbers, target, truncated)

def label(df):
    """As Consistency_Countdown_Screen_Qwen's: the blocks, format_ok, steps_parsed, correct, diagnosis."""
    df = df.copy()
    texts, nums, targets = list(df.completion), [json.loads(x) for x in df.numbers], list(df.target)
    for b in ("reasoning", "solution", "answer"):
        df[b] = [R.tag(t, b) or "" for t in texts]
    df["format_ok"] = [bool(re.search(r"<reasoning>.*?</reasoning>\s*<solution>.*?</solution>\s*<answer>.*?</answer>",
                                      t, re.DOTALL)) for t in texts]
    df["answered"] = df.answer.str.strip() != ""
    df["steps_parsed"] = [bool(R.parse_steps(s)) for s in df.solution]
    df["correct"] = np.array(R.correctness_reward_func(None, texts, nums, targets)) > 0
    df["diagnosis"] = [R.diagnose(a or None, n, t, tr) for a, n, t, tr in zip(df.answer, nums, targets, df.truncated)]
    return df

_p = chat_text([{"role": "user", "content": "x"}])
print("tokenizer:", MODEL, "| pad", tok.pad_token, "| eos", tok.eos_token, "| generation prompt ends:", repr(_p[-50:]))
if CHAT_KWARGS.get("enable_thinking") is False:
    assert _p.rstrip().endswith("</think>"), "thinking-off template should end with an empty think block"


tokenizer: Qwen/Qwen3-0.6B | pad <|endoftext|> | eos <|im_end|> | generation prompt ends: 'im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


In [ ]:
# [puzzles]
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])
_draw = _tz.select(np.random.default_rng(SEED).choice(len(_tz), size=min(len(_tz), 20 * N_PUZZLES), replace=False))
_four = [_as(r) for r in _draw if len(r["nums"]) == 4]
EXAMPLE, puzzles = _four[0], _four[1:N_PUZZLES + 1]
_seen = {_key(p) for p in puzzles} | {_key(EXAMPLE)}
_draw2 = _tz.select(np.random.default_rng(SEED + 1).choice(len(_tz), size=min(len(_tz), 6 * (4000 + 100)), replace=False))
sft_puzzles = []
for r in _draw2:
    p = _as(r)
    if len(p["numbers"]) == 4 and _key(p) not in _seen:
        _seen.add(_key(p))
        sft_puzzles.append(p)
assert len(sft_puzzles) >= N_SFT + N_SFT_VAL, len(sft_puzzles)
sft_puzzles = sft_puzzles[:N_SFT + N_SFT_VAL]
_test_draw = {_key(_as(r)) for r in _draw}
dpo3_puzzles = []                    # the Llama notebook's name: the 3-number puzzles of the SFT draw
for r in _draw2:
    p = _as(r)
    if len(p["numbers"]) == 3 and _key(p) not in _seen and _key(p) not in _test_draw:
        _seen.add(_key(p))
        dpo3_puzzles.append(p)
assert len(dpo3_puzzles) >= N_SFT_3 + 2 * N_SFT_VAL, len(dpo3_puzzles)
sft3_puzzles, sft3_val = dpo3_puzzles[:N_SFT_3], dpo3_puzzles[N_SFT_3:N_SFT_3 + N_SFT_VAL] if N_SFT_3 else []
impossible_puzzles = {4: [], 3: []}

# S's grade puzzles (as the screen): from the grade files; the 4-number ones must be this test draw's
GRADE = {}
for n, f in GRADE_FILES.items():
    g = pd.read_csv(f, keep_default_na=False).drop_duplicates("puzzle").sort_values("puzzle").head(N_GRADE)
    GRADE[n] = [{"puzzle": int(r.puzzle), "numbers": json.loads(r.numbers), "target": int(r.target)} for r in g.itertuples()]
assert [(_key(p)) for p in GRADE[4]] == [_key(p) for p in puzzles[:N_GRADE]], "grade puzzles are not the test draw's"
assert not ({_key(p) for p in sft_puzzles + sft3_puzzles + sft3_val} & {_key(p) for v in GRADE.values() for p in v})
print(f"SFT puzzles: {len(sft_puzzles)} 4-number ({N_SFT} + {N_SFT_VAL} val), {len(sft3_puzzles)} + {len(sft3_val)} "
      f"3-number; grade {({n: len(v) for n, v in GRADE.items()})} (disjoint from the SFT puzzles)")


SFT puzzles: 600 4-number (500 + 100 val), 500 + 100 3-number; grade {4: 200, 3: 200} (disjoint from the SFT puzzles)


In [7]:
# [SFT data] (Consistency_Countdown's, with CHAT_KWARGS per row) One program-written trace per SFT puzzle (traces.py, TRACE_VERSION: 0-3 dead ends, "Back up.", the
# working route, a check; version 2 adds "Left:" on every step, false finishes the check rejects, "Tried:"; solution =
# one step per line; answer = the expression), prompt = the puzzle + format block (PROMPT_STYLE "user"). The last
# N_SFT_VAL traces (per size) give the validation loss. With N_SFT_3, as many 3-number traces too. Saved to SFT_DIR/sft_data.csv. With N_JUDGE_SFT, gold judge-format
# examples (judge_data.py) are mixed into the training set (SFT_DIR/judge_sft_data.csv).
# TRACE_VERSION 3 / 4: FAIL_SHARES (and FAILS_BEFORE_WIN, v4) set the false finishes and attempts per trace, and N_IMPOSSIBLE / N_IMPOSSIBLE_3 unsolvable
# traces (honest stop) go into training only (split "train_impossible" in sft_data.csv).
from datasets import Dataset

_trng = random.Random(SEED)
_v3 = ({"fail_shares": FAIL_SHARES} if TRACE_VERSION == 3 else
       {"fail_shares": FAIL_SHARES, "fails_before_win": FAILS_BEFORE_WIN} if TRACE_VERSION == 4 else {})
sft_traces = [make_trace(p["numbers"], p["target"], _trng, version=TRACE_VERSION, **_v3) for p in sft_puzzles]
sft3_traces = [make_trace(p["numbers"], p["target"], _trng, version=TRACE_VERSION, **_v3) for p in sft3_puzzles + sft3_val]
assert all(check_answer(t["answer"], t["numbers"], t["target"]) for t in sft_traces + sft3_traces), "a trace answer fails the checker"
_irng = random.Random(SEED + 6)
imp_traces = [t for t in (make_trace(p["numbers"], p["target"], _irng, version=TRACE_VERSION, impossible=True, **_v3)
                          for p in impossible_puzzles[4] + impossible_puzzles[3]) if t is not None] if TRACE_VERSION >= 3 else []
assert len(imp_traces) >= 0.95 * (len(impossible_puzzles[4]) + len(impossible_puzzles[3])), len(imp_traces)   # None: too few distinct attempts
_as_row = lambda t: {"prompt": countdown_prompt(t["numbers"], t["target"]),
                     "completion": [{"role": "assistant", "content": t["completion_text"]}],
                     **({"chat_template_kwargs": CHAT_KWARGS} if CHAT_KWARGS else {})}   # TRL's per-example kwargs
_rows, _rows3, _rows_imp = [_as_row(t) for t in sft_traces], [_as_row(t) for t in sft3_traces], [_as_row(t) for t in imp_traces]
if not N_JUDGE_SFT:
    _judge_rows = []
elif N_SFT_3:   # half from each size
    _judge_rows = judge_data.judge_examples(sft_puzzles[:N_JUDGE_SFT // 2] + sft3_puzzles[:N_JUDGE_SFT - N_JUDGE_SFT // 2],
                                            random.Random(SEED + 3))
    _judge_rows = [{**r, **({"chat_template_kwargs": CHAT_KWARGS} if CHAT_KWARGS else {})} for r in _judge_rows]
else:
    _judge_rows = judge_data.judge_examples(sft_puzzles[:min(N_JUDGE_SFT, N_SFT)], random.Random(SEED + 3))
_train_rows = _rows[:N_SFT] + _rows3[:N_SFT_3] + _rows_imp + _judge_rows
random.Random(SEED + 4).shuffle(_train_rows)    # traces (both sizes) and judge examples mixed
sft_dataset, sft_val_dataset = Dataset.from_list(_train_rows), Dataset.from_list(_rows[N_SFT:] + _rows3[N_SFT_3:])
os.makedirs(SFT_DIR, exist_ok=True)
pd.DataFrame(sft_traces + sft3_traces + imp_traces).assign(
    split=["train"] * N_SFT + ["val"] * N_SFT_VAL + ["train"] * N_SFT_3 + ["val"] * len(sft3_val)
          + ["train_impossible"] * len(imp_traces)).to_csv(f"{SFT_DIR}/sft_data.csv", index=False)
_ntok = pd.Series([len(tok(t["completion_text"], add_special_tokens=False)["input_ids"]) for t in sft_traces + sft3_traces + imp_traces])
if _judge_rows:
    pd.DataFrame([{"prompt": r["prompt"][1]["content"], "completion": r["completion"][0]["content"]} for r in _judge_rows]
                 ).to_csv(f"{SFT_DIR}/judge_sft_data.csv", index=False)
    print(f"+ {len(_judge_rows)} gold judge-format examples, e.g.:\n{_judge_rows[0]['completion'][0]['content']}")
print(f"{len(sft_dataset)} train ({len(_judge_rows)} of them judge format, {N_SFT_3} 3-number traces, {len(imp_traces)} unsolvable) + {len(sft_val_dataset)} val traces; completion tokens median {_ntok.median():.0f}, "
      f"max {_ntok.max()}; dead ends: {pd.Series([t['n_dead_ends'] for t in sft_traces]).value_counts().sort_index().to_dict()}; "
      f"false finishes: {pd.Series([t.get('false_finish') for t in sft_traces]).value_counts().to_dict()}"
      + (f"; checks failed per solvable trace: {pd.Series([t['n_check_fails'] for t in sft_traces + sft3_traces]).value_counts().sort_index().to_dict()}, "
         f"per unsolvable: {pd.Series([t['n_check_fails'] for t in imp_traces]).value_counts().sort_index().to_dict()}" if TRACE_VERSION >= 3 else "")
      + (f"; attempts per solvable trace: {pd.Series([t['n_attempts'] for t in sft_traces + sft3_traces]).value_counts().sort_index().to_dict()}, "
         f"per unsolvable: {pd.Series([t['n_attempts'] for t in imp_traces]).value_counts().sort_index().to_dict()}" if TRACE_VERSION == 4 else ""))
print("example:\n" + sft_traces[0]["completion_text"])
if TRACE_VERSION >= 3:
    print("\nexample, two false finishes:\n" + next(t for t in sft_traces if t["n_check_fails"] == 2)["completion_text"])
    if imp_traces:
        print("\nexample, unsolvable:\n" + imp_traces[0]["completion_text"])

1000 train (0 of them judge format, 500 3-number traces, 0 unsolvable) + 200 val traces; completion tokens median 350, max 792; dead ends: {0: 285, 1: 157, 2: 111, 3: 47}; false finishes: {'slip': 80, 'slip,slip': 30, 'gap': 26, 'gap,slip': 14, 'slip,gap': 7, 'gap,gap': 3}; checks failed per solvable trace: {0: 897, 1: 202, 2: 101}, per unsolvable: {}; attempts per solvable trace: {1: 334, 2: 381, 3: 364, 4: 121}, per unsolvable: {}
example:
<reasoning>
Numbers: 46, 42, 63, 50. Target: 64.
Try 63 + 46 = 109. Left: 42, 50, 109.
Then 109 - 42 = 67. Left: 50, 67.
Closest: 67 - 50 = 17, off by 47.
That doesn't work. Back up. Tried: 63 + 46 = 109.
Start again from 46, 42, 63, 50.
Try 63 - 42 = 21. Left: 46, 50, 21.
Then 46 + 50 = 96. Left: 21, 96.
Then 96 - 21 = 64. Left: 64.
That reaches 64.
Check: 63 - 42 = 21 ✓; 46 + 50 = 96 ✓; 96 - 21 = 75, not 64 ✗.
That doesn't work. Back up. Tried: 63 + 46 = 109; 63 - 42 = 21.
Start again from 46, 42, 63, 50.
Try 46 + 50 = 96. Left: 42, 63, 96.
Then 

In [ ]:
# [SFT train] LoRA SFT of the base model on sft_dataset (prompt-completion: TRL applies the chat template with
# CHAT_KWARGS, and puts loss on the assistant turn only, <|eot_id|> included).
# Saves the adapter to SFT_DIR/adapter; validation loss every SFT_EVAL_STEPS, and every SFT_ACC_EVAL_STEPS (and at step
# 0 = the base model) greedy accuracy on the N_SFT_VAL validation puzzles (held out of training and of the test set),
# with the kind breakdown; curve in SFT_DIR/acc_curve.csv, completions in SFT_DIR/acc_eval_stepXXXX.csv.
# Skipped when the adapter exists (RETRAIN = True in [config] to train again).
import gc, torch
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, TrainerCallback

_val_puzzles = sft_puzzles[N_SFT:] if not N_SFT_3 else sft_puzzles[N_SFT:N_SFT + 50] + sft3_val[:50]   # 50 of each size

def accuracy_eval(model, step, batch=50, max_new_tokens=768, out_dir=None):
    """Greedy completions for the validation puzzles with the current weights; accuracy, kinds, truncation."""
    was_training = model.training
    model.eval()
    tok.padding_side = "left"
    rows = []
    for b in range(0, len(_val_puzzles), batch):
        ps = _val_puzzles[b:b + batch]
        texts = [chat_text(countdown_prompt(p["numbers"], p["target"])) for p in ps]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                                 pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for p, o in zip(ps, out[:, enc["input_ids"].shape[1]:]):
            n = int((o != tok.pad_token_id).sum())
            t = tok.decode(o, skip_special_tokens=True)
            truncated = n >= max_new_tokens and tok.eos_token_id not in o.tolist()
            ans = tag(t, "answer")
            rows.append({"step": step, "numbers": json.dumps(p["numbers"]), "target": p["target"], "completion": t,
                         "tokens": n, "truncated": truncated, "answer": ans,
                         "kind": diagnose(ans, p["numbers"], p["target"], truncated)})
    tok.padding_side = "right"
    if was_training:
        model.train()
    df = pd.DataFrame(rows)
    df.to_csv(f"{out_dir or SFT_DIR}/acc_eval_step{step:04d}.csv", index=False)
    return {"step": step, "accuracy": (df.kind == "correct").mean(), "truncated": df.truncated.mean(),
            "tokens": df.tokens.mean(), **{f"kind/{k}": v for k, v in df.kind.value_counts(normalize=True).items()}}

class AccuracyEval(TrainerCallback):
    def __init__(self):
        self.curve = []
    def _run(self, model, step):
        r = accuracy_eval(model, step)
        self.curve.append(r)
        pd.DataFrame(self.curve).to_csv(f"{SFT_DIR}/acc_curve.csv", index=False)
        kinds = ", ".join(f"{k[5:]} {v:.2f}" for k, v in r.items() if k.startswith("kind/") and k != "kind/correct")
        print(f"[accuracy eval] step {step}: accuracy {r['accuracy']:.3f}  truncated {r['truncated']:.2f}  "
              f"tokens {r['tokens']:.0f}  | {kinds}")
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        self._run(model, 0)
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % SFT_ACC_EVAL_STEPS == 0 or state.global_step == state.max_steps:
            self._run(model, state.global_step)

if os.path.exists(f"{SFT_ADAPTER_DIR}/adapter_config.json") and not RETRAIN:
    print(f"[SFT train] skipped: {SFT_ADAPTER_DIR} exists (RETRAIN = False)")
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="auto")
    model.config.pad_token_id = tok.pad_token_id
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, bias="none", task_type="CAUSAL_LM",
                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    _total_steps = -(-len(sft_dataset) // (SFT_BATCH * SFT_GRAD_ACC)) * SFT_EPOCHS
    sft_args = SFTConfig(
        output_dir=f"{SFT_DIR}/checkpoints",
        num_train_epochs=SFT_EPOCHS, learning_rate=SFT_LR, lr_scheduler_type="cosine",
        warmup_steps=max(1, round(0.05 * _total_steps)),
        per_device_train_batch_size=SFT_BATCH, gradient_accumulation_steps=SFT_GRAD_ACC,
        max_length=1024,                  # prompt (~120 tokens) + trace (a few hundred); nothing is cut
        bf16=True, gradient_checkpointing=True,
        logging_steps=10, save_strategy="no", report_to="none", seed=SEED,
        eval_strategy="steps", eval_steps=SFT_EVAL_STEPS, per_device_eval_batch_size=SFT_BATCH,
    )
    print(f"{len(sft_dataset)} examples, {_total_steps} optimizer steps")
    trainer = SFTTrainer(model=model, args=sft_args, train_dataset=sft_dataset, eval_dataset=sft_val_dataset,
                         processing_class=tok, peft_config=lora_config, callbacks=[AccuracyEval()])

    # Check tokenized examples: one BOS, pinned date, loss only on the completion, ending in <|eot_id|>; a trace starts
    # with <reasoning>, a judge-format example is the neutral system prompt + puzzle -> steps + "Answer: ...".
    _kinds = set()
    for _k in range(min(200, len(trainer.train_dataset))):
        _row = trainer.train_dataset[_k]
        _ids = _row["input_ids"]
        _lab = _row.get("labels") or [i if m else -100 for i, m in zip(_ids, _row["completion_mask"])]
        _comp, _full = tok.decode([i for i, l in zip(_ids, _lab) if l != -100]), tok.decode(_ids)
        assert tok.bos_token_id is None or _ids.count(tok.bos_token_id) == 1, "BOS count"
        assert _comp.rstrip().endswith(tok.eos_token), _comp[-40:]      # Qwen3: "<|im_end|>\n"
        if CHAT_KWARGS.get("enable_thinking") is False:
            assert "<think>" not in _comp and _full.count("<think>\n\n</think>") == 1, _full[-400:]
        if _comp.startswith("<reasoning>"):
            _kinds.add("trace")
        else:
            assert prompts.NEUTRAL_SYSTEM_PROMPT in _full and "\nAnswer: " in _comp and "<reasoning>" not in _full, _full[:300]
            _kinds.add("judge")
    assert _kinds == ({"trace", "judge"} if N_JUDGE_SFT else {"trace"}), _kinds
    print("example, tokenized:\n" + _full[-600:])
    print(f"tokenized ok ({sorted(_kinds)} in the first 200): e.g. {len(_ids)} tokens, {sum(l != -100 for l in _lab)} with loss")
    assert max(len(r["input_ids"]) for r in trainer.train_dataset) <= sft_args.max_length

    trainer.train()
    trainer.model.save_pretrained(SFT_ADAPTER_DIR)   # adapter only: vLLM would load a tokenizer found in the LoRA folder
    # the [config] values and trainer settings of this SFT (the folder name doesn't carry all of them)
    try:   # a record only: never fail the cell after training
        json.dump({**{k: v for k, v in globals().items() if k.isupper() and not k.startswith("_")
                      and isinstance(v, (bool, int, float, str, tuple, list, dict, type(None)))},
                   "sft_args": sft_args.to_dict()}, open(f"{SFT_DIR}/config.json", "w"), indent=1, default=str)
    except Exception as e:
        print(f"[config.json] not written: {e!r}")
    _log = pd.DataFrame(trainer.state.log_history)
    _log.to_csv(f"{SFT_DIR}/train_log.csv", index=False)
    print("saved adapter to", SFT_ADAPTER_DIR)
    print(_log.dropna(subset=["eval_loss"])[["step", "eval_loss"]].to_string(index=False))
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

1000 examples, 63 optimizer steps


Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

... (3 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

example, tokenized:
<think>

</think>

<reasoning>
Numbers: 13, 55, 59, 96. Target: 31.
Try 59 * 55 = 3245. Left: 13, 96, 3245.
Then 3245 - 96 = 3149. Left: 13, 3149.
Closest: 3149 - 13 = 3136, off by 3105.
That doesn't work. Back up. Tried: 59 * 55 = 3245.
Start again from 13, 55, 59, 96.
Try 96 - 13 = 83. Left: 55, 59, 83.
Then 83 - 59 = 24. Left: 55, 24.
Then 55 - 24 = 31. Left: 31.
That reaches 31.
Check: 96 - 13 = 83 ✓; 83 - 59 = 24 ✓; 55 - 24 = 31 ✓; 13, 55, 59, 96 used once each ✓.
</reasoning>
<solution>
96 - 13 = 83
83 - 59 = 24
55 - 24 = 31
</solution>
<answer>
55 - ((96 - 13) - 59)
</answer><|im_end|>

tokenized ok (['trace'] in the first 200): e.g. 494 tokens, 361 with loss
[accuracy eval] step 0: accuracy 0.010  truncated 0.19  tokens 286  | no answer 0.73, no answer (truncated) 0.19, right numbers, wrong value 0.04, wrong numbers (made up or swapped) 0.01, not an expression 0.01, a number missing 0.01


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
25,0.189493,0.080779,0.089863,0.970794,198543.000000
50,0.054839,0.053273,0.052481,0.977184,391198.000000
63,0.051953,0.052340,0.052292,0.977575,490863.000000


[accuracy eval] step 25: accuracy 0.170  truncated 0.35  tokens 419  | right numbers, wrong value 0.43, no answer (truncated) 0.35, a number missing 0.03, wrong numbers (made up or swapped) 0.02
[accuracy eval] step 50: accuracy 0.230  truncated 0.63  tokens 584  | no answer (truncated) 0.63, right numbers, wrong value 0.07, a number reused or added 0.04, wrong numbers (made up or swapped) 0.02, a number missing 0.01
[accuracy eval] step 63: accuracy 0.250  truncated 0.61  tokens 580  | no answer (truncated) 0.61, a number reused or added 0.06, right numbers, wrong value 0.05, wrong numbers (made up or swapped) 0.02, a number missing 0.01
saved adapter to /content/drive/MyDrive/Consistency/countdown/sft/Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500/adapter
 step  eval_loss
   25   0.080779
   50   0.053273
   63   0.052340
GPU free: 37.8 GiB


In [ ]:
# [grade] GRADE_SAMPLES per grade puzzle and size at T TEMPERATURE, cap MAX_TOKENS, seed SEED, the GRPO prompt (CHAT_KWARGS),
# from the base model + the SFT adapter in vLLM; labeled as the screen, saved to GRADE_DIR/gen_sft.csv; then the
# screen's table per size and cap (finished at cap C = stopped within C tokens), with the untrained rows (SCREEN).
import torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
os.makedirs(GRADE_DIR, exist_ok=True)
_gen = f"{GRADE_DIR}/gen_sft.csv"
if os.path.exists(_gen):
    print(f"[grade] {_gen} exists, skipped")
else:
    assert os.path.exists(f"{SFT_ADAPTER_DIR}/adapter_config.json"), "no adapter: run [SFT train]"
    t0 = time.time()
    batch = [(n, p) for n in (4, 3) for p in GRADE[n]]
    ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for n, p in batch]
    llm = LLM(model=MODEL, dtype="bfloat16", gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION, enable_lora=True,
              max_lora_rank=LORA_R, max_model_len=max(map(len, ids)) + MAX_TOKENS + 16, seed=SEED)
    outs = llm.generate([{"prompt_token_ids": x} for x in ids],
                        SamplingParams(n=GRADE_SAMPLES, temperature=TEMPERATURE, top_p=1.0, max_tokens=MAX_TOKENS, seed=SEED),
                        lora_request=LoRARequest("sft", 1, SFT_ADAPTER_DIR))
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    rows = [{"model": f"{MODEL_KEY}-sft", "n_numbers": n, "puzzle": p["puzzle"], "numbers": json.dumps(p["numbers"]),
             "target": p["target"], "sample": k, "completion": c.text, "n_tokens": len(c.token_ids),
             "finished": c.finish_reason == "stop", "truncated": c.finish_reason != "stop"}
            for (n, p), o in zip(batch, outs) for k, c in enumerate(o.outputs)]
    label(pd.DataFrame(rows)).to_csv(_gen, index=False)
    print(f"[grade] {len(rows)} completions in {(time.time() - t0) / 60:.1f} min; saved {_gen}")
    del llm
    gc.collect()
    torch.cuda.empty_cache()

d = pd.read_csv(_gen, keep_default_na=False, dtype={c: str for c in ("completion", "reasoning", "solution", "answer")})
for c in ("finished", "truncated", "format_ok", "answered", "steps_parsed", "correct"):
    d[c] = d[c].astype(str) == "True"

def stats(d, cap):
    """As the screen's [summary]."""
    fin = d.finished & (d.n_tokens <= cap)
    ans, ok = fin & d.answered, fin & d.correct
    g = d.assign(aw=ans & ~ok, ok=ok).groupby("puzzle")
    return {"n": len(d), "accuracy": ok.mean(), "format ok": (fin & d.format_ok).mean(),
            "solution parses": (fin & d.steps_parsed).mean(), "truncated": 1 - fin.mean(),
            "finished, no answer": (fin & ~d.answered).mean(), "answered wrong": (ans & ~ok).mean(),
            "correct among answered": ok.sum() / max(ans.sum(), 1), "tokens (median, finished)": d.n_tokens[fin].median(),
            "puzzles: correct + answered wrong": int((g.ok.any() & g.aw.any()).sum()), "pass@k": g.ok.any().mean()}

summary = pd.DataFrame([{"model": f"{MODEL_KEY}-sft", "numbers": n, "cap": c, **stats(dd, c)}
                        for n, dd in d.groupby("n_numbers") for c in CAPS])
summary.to_csv(f"{GRADE_DIR}/summary.csv", index=False)
_both = pd.concat([pd.read_csv(SCREEN).query("model == @MODEL_KEY") if os.path.exists(SCREEN) else None, summary])
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
    print(f"[grade] S's grade puzzles, {N_GRADE} x {GRADE_SAMPLES} per size, T {TEMPERATURE:g}, seed {SEED}, GRPO prompt, "
          f"thinking off; untrained rows from the screen\n" + _both.sort_values(["numbers", "cap", "model"]).to_string(index=False))
w = d[d.answered & d.finished & ~d.correct]
print("\nanswered-wrong classes (cap 2048):", "; ".join(f"{c} {v}" for c, v in Counter(w.diagnosis).most_common(6)))
for lab, sub in (("correct", d[d.correct]), ("answered wrong", w)):
    if len(sub):
        r = sub.sample(1, random_state=SEED).iloc[0]
        print(f"\n--- {lab}: {r.numbers} -> {r.target}, {r.n_tokens} tokens\n{r.completion[-900:]}")


In [ ]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE.
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()
